In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

In [2]:
PROJECT_ROOT = Path.cwd().parent

PROCESSED_DIR = PROJECT_ROOT / "DATA" / "PROCESSED"

CLEAN_FILE = PROCESSED_DIR / "cars_clean.csv"
FEATURE_FILE = PROCESSED_DIR / "cars_features.csv"

In [3]:
cars = pd.read_csv(CLEAN_FILE)

print("Shape:", cars.shape)
cars.head()

Shape: (1725, 9)


,id,year,brand,full_model_name,model_name,price,distance_travelled(kms),fuel_type,city
0,0,2016,Honda,Honda Brio S MT,Brio,425000.0,9680.0,Petrol,Mumbai
1,1,2012,Nissan,Nissan Sunny XV Diesel,Sunny,325000.0,119120.0,Diesel,Mumbai
2,2,2017,Toyota,Toyota Fortuner 2.8 4x2 MT [2016-2020],Fortuner,2650000.0,64593.0,Diesel,Thane
3,3,2017,Mercedes-Benz,Mercedes-Benz E-Class E 220d Expression [2019-...,E-Class,4195000.0,25000.0,Diesel,Mumbai
4,4,2012,Hyundai,Hyundai Verna Fluidic 1.6 CRDi SX,Verna,475000.0,23800.0,Diesel,Mumbai


In [4]:
price_candidates = [
    "selling_price",
    "price",
    "resale_price"
]

target_candidates = [
    col for col in price_candidates
    if col in cars.columns
]

if not target_candidates:
    raise ValueError("No price column found.")

TARGET = target_candidates[0]

print("Target column:", TARGET)

Target column: price


In [5]:
from datetime import datetime

CURRENT_YEAR = datetime.now().year

if "year" in cars.columns:

    cars["car_age"] = (
        CURRENT_YEAR - cars["year"]
    )

    cars["car_age"] = cars["car_age"].clip(lower=0)

    print(cars[["year", "car_age"]].head())

   year  car_age
0  2016       10
1  2012       14
2  2017        9
3  2017        9
4  2012       14


In [6]:
cars["log_price"] = np.log1p(cars[TARGET])

cars[[TARGET, "log_price"]].head()

,price,log_price
0,425000.0,12.959847
1,325000.0,12.691584
2,2650000.0,14.790071
3,4195000.0,15.249404
4,475000.0,13.071072


In [7]:
numeric_columns = cars.select_dtypes(
    include=np.number
).columns

for column in numeric_columns:
    cars[column] = pd.to_numeric(
        cars[column],
        errors="coerce"
    )

print("Numeric columns processed.")

Numeric columns processed.


In [8]:
numeric_columns = cars.select_dtypes(include=np.number).columns

for column in numeric_columns:
    cars[column] = cars[column].fillna(
        cars[column].median()
    )

categorical_columns = cars.select_dtypes(
    include=["object", "category"]
).columns

for column in categorical_columns:
    mode = cars[column].mode()

    if len(mode) > 0:
        cars[column] = cars[column].fillna(
            mode.iloc[0]
        )
    else:
        cars[column] = cars[column].fillna("Unknown")

In [9]:
# We don't want log_price to be used as a feature
# because it is directly calculated from the target.

DROP_COLUMNS = [
    "log_price"
]

DROP_COLUMNS = [
    col for col in DROP_COLUMNS
    if col in cars.columns
]

model_data = cars.drop(
    columns=DROP_COLUMNS
)

print("Dropped columns:", DROP_COLUMNS)
print("Final feature-engineering shape:", model_data.shape)

Dropped columns: ['log_price']
Final feature-engineering shape: (1725, 10)


In [10]:
print("Final columns:")

for column in model_data.columns:
    print("-", column)

Final columns:
- id
- year
- brand
- full_model_name
- model_name
- price
- distance_travelled(kms)
- fuel_type
- city
- car_age


In [11]:
model_data.to_csv(
    FEATURE_FILE,
    index=False
)

print("Feature dataset saved:")
print(FEATURE_FILE)

Feature dataset saved:
C:\Users\sitar\Downloads\ASSIGNMENT_CODE\CARS\DATA\PROCESSED\cars_features.csv
